In [38]:
import duckdb
from pathlib import Path

In [39]:
# See what is actually in data/processed/
for p in Path("data/processed").rglob("*.parquet"):
    print(p, f"{p.stat().st_size/1e6:.1f} MB")

data\processed\casualties.parquet 7.5 MB
data\processed\collisions.parquet 15.5 MB
data\processed\imd_2025.parquet 1.1 MB
data\processed\vehicles.parquet 16.8 MB


In [40]:
con = duckdb.connect("warehouse.duckdb")

In [41]:
for name in ["collisions", "vehicles", "casualties", "imd_2025"]:
    print("=" * 60)
    print(name)
    print("=" * 60)
    df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('data/processed/{name}.parquet')").df()
    print(df[["column_name", "column_type"]].to_string(index=False))

collisions
                 column_name column_type
             collision_index     VARCHAR
              collision_year      BIGINT
                        date   TIMESTAMP
                        year     INTEGER
                       month     INTEGER
                        hour      BIGINT
                police_force      DOUBLE
    local_authority_district      DOUBLE
local_authority_ons_district     VARCHAR
                   longitude      DOUBLE
                    latitude      DOUBLE
                   road_type      DOUBLE
         urban_or_rural_area      DOUBLE
            light_conditions      DOUBLE
          weather_conditions      DOUBLE
     road_surface_conditions      DOUBLE
            first_road_class      DOUBLE
                 speed_limit      DOUBLE
             junction_detail      DOUBLE
          number_of_vehicles      BIGINT
        number_of_casualties      BIGINT
          collision_severity      DOUBLE
vehicles
                     column_name colu

In [42]:
def run_sql(p): return con.execute(Path(p).read_text(encoding="utf-8"))
def read_sql(p): return con.sql(Path(p).read_text(encoding="utf-8")).df()

# one-time
for f in sorted(Path("sql/build").glob("*.sql")):
    print("→", f.name)
    run_sql(f)


→ 00_init.sql
→ 01_dimensions.sql
→ 02_facts.sql
→ 03_load.sql


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

→ 04_dim_date.sql
→ 05_dim_local_authority.sql
→ 06_views.sql
→ 07_qa_checks.sql


In [43]:
con.sql("SHOW ALL TABLES").df() 

,database,schema,name,column_names,column_types,temporary
0,warehouse,stats19,dim_casualty_class,"[code, label]","[SMALLINT, VARCHAR]",False
1,warehouse,stats19,dim_date,"[date_key, year, quarter, month, month_name, d...","[DATE, SMALLINT, TINYINT, TINYINT, VARCHAR, TI...",False
2,warehouse,stats19,dim_imd,"[lsoa_code, lsoa_name, lad_code, lad_name, imd...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, DOUBLE, I...",False
3,warehouse,stats19,dim_local_authority,"[lad_code, lad_name, region, highway_authority]","[VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
4,warehouse,stats19,dim_severity,"[code, label]","[SMALLINT, VARCHAR]",False
5,warehouse,stats19,dim_sex,"[code, label]","[SMALLINT, VARCHAR]",False
6,warehouse,stats19,dim_urban_rural,"[code, label]","[SMALLINT, VARCHAR]",False
7,warehouse,stats19,fact_casualties,"[collision_index, vehicle_reference, casualty_...","[VARCHAR, SMALLINT, SMALLINT, TINYINT, TINYINT...",False
8,warehouse,stats19,fact_collisions,"[collision_index, collision_year, date, year, ...","[VARCHAR, SMALLINT, DATE, SMALLINT, TINYINT, T...",False
9,warehouse,stats19,fact_vehicles,"[collision_index, vehicle_reference, vehicle_t...","[VARCHAR, SMALLINT, TINYINT, TINYINT, TINYINT,...",False


In [44]:
# analysis
read_sql("sql/analysis/a01_severity_by_imd.sql")

,imd_decile,killed,serious,slight,total,ksi_pct
0,1,656,13573,52004,66233,21.48
1,2,604,12016,53349,65969,19.13
2,3,537,11711,52588,64836,18.89
3,4,632,10928,48190,59750,19.35
4,5,583,10516,44323,55422,20.03
5,6,614,9814,39529,49957,20.87
6,7,585,9071,37183,46839,20.62
7,8,466,8514,33519,42499,21.13
8,9,530,7993,31707,40230,21.19
9,10,380,6951,27419,34750,21.10


In [45]:
read_sql("sql/analysis/a02_yearly_trend.sql")

,year,collisions,casualties,killed,serious,slight
0,2021,87407,108876,1143,19187,88546
1,2022,90873,113539,1224,21031,91284
2,2023,88482,110166,1149,20787,88230
3,2024,84396,104770,1092,20520,83158
4,2025,86672,106932,1109,22570,83253


In [46]:
read_sql("sql/analysis/a03_hotspots_la.sql")

,lad_name,collisions,casualties,ksi,ksi_pct
0,Leeds,6895,8233,2119,25.74
1,Birmingham,9657,11648,1988,17.07
2,Bradford,5750,7206,1671,23.19
3,North Yorkshire,4236,5104,1143,22.39
4,Kirklees,3749,4484,1120,24.98
5,Cornwall,4465,5675,1114,19.63
6,Wakefield,3100,3742,954,25.49
7,Lambeth,5046,5262,924,17.56
8,Doncaster,2672,3285,917,27.91
9,Somerset,3810,4702,903,19.20


In [47]:
read_sql("sql/analysis/a04_age_groups.sql")

,age_band,casualties,ksi
0,0–15,45960,8692
1,16–24,103719,21638
2,25–39,162278,28050
3,40–59,143672,27894
4,60–74,54173,14271
5,75+,34481,9267


In [48]:
read_sql("sql/analysis/a05_urban_rural.sql")

,urban_rural,hour,casualties
0,Rural,0,3137
1,Rural,1,2087
2,Rural,2,1546
3,Rural,3,1266
4,Rural,4,1250
...,...,...,...
59,NaN,19,2
60,NaN,20,5
61,NaN,21,2
62,NaN,22,1


In [49]:
export_dir = Path("data/powerbi")
export_dir.mkdir(parents=True, exist_ok=True)

tables = [
    "dim_imd", "dim_local_authority", "dim_date",
    "dim_severity", "dim_casualty_class", "dim_urban_rural", "dim_sex",
    "fact_collisions", "fact_vehicles", "fact_casualties",
    "v_collisions", "v_casualties_enriched",
]
for t in tables:
    out = (export_dir / f"{t}.parquet").as_posix()
    con.execute(f"COPY (SELECT * FROM stats19.{t}) TO '{out}' (FORMAT PARQUET)")
    print("exported:", out)

exported: data/powerbi/dim_imd.parquet
exported: data/powerbi/dim_local_authority.parquet
exported: data/powerbi/dim_date.parquet
exported: data/powerbi/dim_severity.parquet
exported: data/powerbi/dim_casualty_class.parquet
exported: data/powerbi/dim_urban_rural.parquet
exported: data/powerbi/dim_sex.parquet
exported: data/powerbi/fact_collisions.parquet
exported: data/powerbi/fact_vehicles.parquet
exported: data/powerbi/fact_casualties.parquet
exported: data/powerbi/v_collisions.parquet
exported: data/powerbi/v_casualties_enriched.parquet
